# Reranker Step 2 — Feature Engineering (v3)

## 1. Configuration

In [1]:
import os

# ── Paths ──────────────────────────────────────────────────────────────
PHASE1_DIR = "phase1_outputs"
PHASE3_DIR = "phase3_outputs"

TRAIN_FILE            = os.path.join(PHASE1_DIR, "interactions_model_ready_train.parquet")
ITEM_META_FILE        = os.path.join(PHASE1_DIR, "item_meta.parquet")
VALID_CANDIDATES_FILE = os.path.join(PHASE3_DIR, "valid_candidates.parquet")
TEST_CANDIDATES_FILE  = os.path.join(PHASE3_DIR, "test_candidates.parquet")

VALID_FEATURES_FILE = os.path.join(PHASE3_DIR, "valid_lambdamart_features.parquet")
TEST_FEATURES_FILE  = os.path.join(PHASE3_DIR, "test_lambdamart_features.parquet")

# ── Feature construction parameters ────────────────────────────────────
# Bayesian damping: shrink per-hotel averages toward the global mean when
# the hotel has few ratings. C=10 means "10 virtual ratings at the global mean".
BAYESIAN_PRIOR_STRENGTH = 10

# Cofreq normalization smoothing — avoids division by near-zero.
COFREQ_NORM_EPS = 1.0

# Per-aspect averages we keep, mapped to feature names produced after damping.
# Reduced from six to three (value, location, sleep) based on v2 diagnostics:
# rooms, cleanliness, and service all corr > 0.87 with these or with bayesian_avg.
ASPECT_TO_FEATURE = {
    "avg_value":         "hotel_avg_value_damped",
    "avg_location":      "hotel_avg_location_damped",
    "avg_sleep_quality": "hotel_avg_sleep_damped",
}

# Hotel aspect feature columns, used by §5 to derive user aspect preferences
# and by §6 to compute deltas.
HOTEL_ASPECT_COLS = list(ASPECT_TO_FEATURE.values())
USER_ASPECT_COLS  = ["user_avg_value", "user_avg_location", "user_avg_sleep"]
DELTA_COLS        = ["delta_value", "delta_location", "delta_sleep"]

# ── Diagnostic parameters ─────────────────────────────────────────────
# NDCG@10 is the reporting metric. Diagnostics use the same K so they stay
# aligned with what the final model is judged on.
UNIVARIATE_NDCG_K = 10
N_CORR_TOP_PAIRS  = 15


## 2. Imports

In [2]:
import time
import numpy as np
import pandas as pd
import scipy.sparse as sp

pd.set_option("display.float_format", lambda x: f"{x:.4f}")
pd.set_option("display.max_columns", 40)


## 3. Load inputs

- **Candidate files**
- **Train interactions:**
- **Item metadata:**


In [3]:
t0 = time.time()

# Candidates
valid_candidates_df = pd.read_parquet(VALID_CANDIDATES_FILE)
test_candidates_df  = pd.read_parquet(TEST_CANDIDATES_FILE)
print(f"Valid candidates: {valid_candidates_df.shape}  cols={list(valid_candidates_df.columns)}")
print(f"Test candidates:  {test_candidates_df.shape}")

# Train interactions — filter to train split defensively
train_raw = pd.read_parquet(TRAIN_FILE, columns=["user_idx", "item_idx", "rating", "split"])
n_before = len(train_raw)
train_df = train_raw[train_raw["split"] == "train"][["user_idx", "item_idx", "rating"]].reset_index(drop=True)
n_filtered = n_before - len(train_df)
if n_filtered > 0:
    print(f"  filtered out {n_filtered:,} non-train rows")
del train_raw

global_mean_rating = float(train_df["rating"].mean())

# Item metadata
item_meta_raw = pd.read_parquet(ITEM_META_FILE)
ASPECT_COLS = list(ASPECT_TO_FEATURE.keys())
required_cols = ["item_idx", "geo_id", "train_review_count", "bayesian_avg",
                 "n_reviews_with_any_subrating"] + ASPECT_COLS
missing = [c for c in required_cols if c not in item_meta_raw.columns]
if missing:
    raise ValueError(f"item_meta missing required columns: {missing}")

print(f"\nTrain rows:        {len(train_df):,}")
print(f"Train users:       {train_df['user_idx'].nunique():,}")
print(f"Train items:       {train_df['item_idx'].nunique():,}")
print(f"Global mean rating: {global_mean_rating:.4f}")
print(f"Item meta:         {item_meta_raw.shape}")
print(f"\nTotal load time: {time.time()-t0:.1f}s")


Valid candidates: (50000000, 6)  cols=['user_idx', 'candidate_item_idx', 'lightgcn_score', 'rank', 'true_item_idx', 'is_positive']
Test candidates:  (25000000, 6)

Train rows:        16,064,988
Train users:       1,927,834
Train items:       309,395
Global mean rating: 4.0711
Item meta:         (309395, 16)

Total load time: 1.0s


## 4. Hotel (item) features

- **`hotel_review_count_log`** — `log1p(train_review_count)`. Long-tail compression.
- **`hotel_bayesian_avg`** — overall rating, taken directly from `item_meta.bayesian_avg` (already damped upstream).
- **Three aspect averages** (value, location, sleep), each Bayesian-damped here using `n_reviews_with_any_subrating` as the support count. Hotels with no sub-ratings are filled with the per-aspect global mean before damping, which collapses the formula to `mu_global` for them.
- **`hotel_popularity_in_geo`** — `log1p(hotel_review_count / total_reviews_in_geo)`. Distinguishes globally-medium hotels that dominate their local market from globally-medium hotels in saturated markets.


In [4]:
t0 = time.time()

# ── Per-aspect global means (weighted by n_reviews_with_any_subrating) ─
aspect_global_means = {}
for asp in ASPECT_COLS:
    mask = item_meta_raw[asp].notna() & (item_meta_raw["n_reviews_with_any_subrating"] > 0)
    if mask.sum() == 0:
        aspect_global_means[asp] = global_mean_rating
    else:
        num = (item_meta_raw.loc[mask, asp] * item_meta_raw.loc[mask, "n_reviews_with_any_subrating"]).sum()
        den = item_meta_raw.loc[mask, "n_reviews_with_any_subrating"].sum()
        aspect_global_means[asp] = float(num / den)

print("Per-aspect global means (for Bayesian damping):")
for asp, mu in aspect_global_means.items():
    print(f"  {asp:18s}: {mu:.4f}")

# ── Build item_features ────────────────────────────────────────────────
item_features = item_meta_raw[["item_idx", "geo_id", "train_review_count", "bayesian_avg"]].copy()
item_features["hotel_review_count_log"] = np.log1p(item_features["train_review_count"]).astype(np.float32)
item_features = item_features.rename(columns={"bayesian_avg": "hotel_bayesian_avg"})
item_features["hotel_bayesian_avg"] = item_features["hotel_bayesian_avg"].astype(np.float32)

C = BAYESIAN_PRIOR_STRENGTH
n_sub = item_meta_raw["n_reviews_with_any_subrating"].fillna(0).astype(float)
for asp, out_name in ASPECT_TO_FEATURE.items():
    mu_g = aspect_global_means[asp]
    raw  = item_meta_raw[asp].fillna(mu_g)
    damped = (raw * n_sub + C * mu_g) / (n_sub + C)
    item_features[out_name] = damped.astype(np.float32)

# ── Hotel popularity within its own geo ────────────────────────────────
geo_totals = (
    item_meta_raw.groupby("geo_id")["train_review_count"].sum()
    .rename("geo_total_reviews").reset_index()
)
item_features = item_features.merge(geo_totals, on="geo_id", how="left")
item_features["hotel_popularity_in_geo"] = np.log1p(
    item_features["train_review_count"] / item_features["geo_total_reviews"].clip(lower=1)
).astype(np.float32)
item_features = item_features.drop(columns=["geo_total_reviews"])

print(f"\nItem features: {item_features.shape}  in {time.time()-t0:.1f}s")
print(item_features.drop(columns=["item_idx", "geo_id", "train_review_count"]).describe().round(4))


Per-aspect global means (for Bayesian damping):
  avg_value         : 4.0502
  avg_location      : 4.3393
  avg_sleep_quality : 4.1494

Item features: (309395, 9)  in 0.1s
       hotel_bayesian_avg  hotel_review_count_log  hotel_avg_value_damped  \
count         309395.0000             309395.0000             309395.0000   
mean               4.0353                  3.1230                  4.0437   
std                0.3195                  1.1996                  0.3741   
min                1.8727                  0.6931                  1.7625   
25%                3.8570                  2.1972                  3.8334   
50%                4.0613                  2.9444                  4.0502   
75%                4.2400                  3.9512                  4.3149   
max                4.9487                  8.3214                  4.9395   

       hotel_avg_location_damped  hotel_avg_sleep_damped  \
count                309395.0000             309395.0000   
mean          

## 5. User features

**Profile (3):**
- **`user_avg_rating`** — generosity baseline.
- **`user_review_count_log`** — power-user vs. one-shot reviewer.
- **`user_rating_std`** — discrimination / decisiveness. Single-review users get `0.0`.

**Aspect preferences (3, NEW in v3):**
- **`user_avg_value`** = mean of `hotel_avg_value_damped` across the user's training hotels.
- **`user_avg_location`** = mean of `hotel_avg_location_damped` across the user's training hotels.
- **`user_avg_sleep`** = mean of `hotel_avg_sleep_damped` across the user's training hotels.


In [5]:
t0 = time.time()

# ── Basic rating stats ────────────────────────────────────────────────
user_rating_stats = train_df.groupby("user_idx").agg(
    user_review_count=("rating", "size"),
    user_avg_rating=("rating", "mean"),
    user_rating_std=("rating", "std"),
).reset_index()
# Single-review users have NaN std → 0 (no observed variation)
user_rating_stats["user_rating_std"] = user_rating_stats["user_rating_std"].fillna(0.0)
user_rating_stats["user_review_count_log"] = np.log1p(user_rating_stats["user_review_count"])

# ── Average log-popularity of items the user has interacted with ──────
# Used only as an intermediate for `candidate_pop_minus_user_avg_pop` in §6.
train_with_pop = train_df.merge(
    item_features[["item_idx", "hotel_review_count_log"]],
    on="item_idx", how="left",
)
user_pop_stats = (
    train_with_pop.groupby("user_idx")["hotel_review_count_log"]
                  .mean()
                  .reset_index()
                  .rename(columns={"hotel_review_count_log": "user_avg_item_pop_log"})
)
del train_with_pop

# ── User aspect preferences (NEW in v3) ───────────────────────────────
# For each user, average the hotel aspect averages over their training history.
# This is the closest available proxy to "what aspect levels does this user
# tend to visit" given that train reviews don't expose per-review aspect ratings.
train_with_aspects = train_df.merge(
    item_features[["item_idx"] + HOTEL_ASPECT_COLS],
    on="item_idx", how="left",
)
user_aspects = train_with_aspects.groupby("user_idx")[HOTEL_ASPECT_COLS].mean().reset_index()
user_aspects.columns = ["user_idx"] + USER_ASPECT_COLS
for col in USER_ASPECT_COLS:
    user_aspects[col] = user_aspects[col].astype(np.float32)
del train_with_aspects

# ── Combine ───────────────────────────────────────────────────────────
user_features = (
    user_rating_stats
    .merge(user_pop_stats,  on="user_idx", how="left")
    .merge(user_aspects,    on="user_idx", how="left")
)

USER_FEATURE_COLS = (
    ["user_idx", "user_avg_rating", "user_review_count_log", "user_rating_std"]
    + USER_ASPECT_COLS
    + ["user_avg_item_pop_log"]   # intermediate, dropped after cross feature is built
)
user_features = user_features[USER_FEATURE_COLS]

print(f"User features: {user_features.shape}  in {time.time()-t0:.1f}s")
print(user_features.drop(columns=["user_idx"]).describe().round(4))


User features: (1927834, 8)  in 3.2s
       user_avg_rating  user_review_count_log  user_rating_std  \
count     1927834.0000           1927834.0000     1927834.0000   
mean            4.1046                 2.0075           0.8450   
std             0.5672                 0.5990           0.4792   
min             1.0000                 1.3863           0.0000   
25%             3.7500                 1.6094           0.5477   
50%             4.1500                 1.7918           0.8165   
75%             4.5000                 2.3026           1.1362   
max             5.0000                 6.3613           2.3094   

       user_avg_value  user_avg_location  user_avg_sleep  \
count    1927834.0000       1927834.0000    1927834.0000   
mean           4.0584             4.3524          4.1573   
std            0.1984             0.2053          0.2308   
min            2.6121             2.9076          2.5685   
25%            3.9384             4.2256          4.0176   
50%     

## 6. Base feature assembly

Joins item and user features onto the candidate dataframes, computes the per-user score z-score, and builds the cross + delta feature


In [6]:
BASE_COLS = ["user_idx", "candidate_item_idx", "true_item_idx", "is_positive"]

ITEM_JOIN_COLS = [
    "item_idx",  # join key
    "hotel_review_count_log",
    "hotel_bayesian_avg",
    "hotel_avg_value_damped",
    "hotel_avg_location_damped",
    "hotel_avg_sleep_damped",
    "hotel_popularity_in_geo",
    "geo_id",  # carried as candidate_geo_id, used by §9 then dropped
]


def build_base_features(candidates_df, split_name):
    t0 = time.time()
    df = candidates_df.rename(columns={"rank": "lightgcn_rank"}).copy()

    # ── Per-user score z-score ────────────────────────────────────────
    score_stats = (
        df.groupby("user_idx")["lightgcn_score"]
          .agg(["mean", "std"])
          .rename(columns={"mean": "_u_score_mean", "std": "_u_score_std"})
          .reset_index()
    )
    score_stats["_u_score_std"] = (
        score_stats["_u_score_std"].replace(0, np.nan).fillna(1.0)
    )
    df = df.merge(score_stats, on="user_idx", how="left")
    df["lightgcn_score_zscore"] = (
        (df["lightgcn_score"] - df["_u_score_mean"]) / df["_u_score_std"]
    ).astype(np.float32)
    df = df.drop(columns=["_u_score_mean", "_u_score_std"])

    # ── Item features (also brings candidate_geo_id) ──────────────────
    df = df.merge(
        item_features[ITEM_JOIN_COLS]
            .rename(columns={"item_idx": "candidate_item_idx", "geo_id": "candidate_geo_id"}),
        on="candidate_item_idx", how="left",
    )

    # ── User features ─────────────────────────────────────────────────
    df = df.merge(user_features, on="user_idx", how="left")

    # ── Cross features ────────────────────────────────────────────────
    df["candidate_rating_minus_user_avg"] = (
        df["hotel_bayesian_avg"] - df["user_avg_rating"]
    ).astype(np.float32)
    df["candidate_pop_minus_user_avg_pop"] = (
        df["hotel_review_count_log"] - df["user_avg_item_pop_log"]
    ).astype(np.float32)
    df = df.drop(columns=["user_avg_item_pop_log"])

    # ── Aspect deltas (NEW in v3) ─────────────────────────────────────
    df["delta_value"]    = (df["hotel_avg_value_damped"]    - df["user_avg_value"]   ).astype(np.float32)
    df["delta_location"] = (df["hotel_avg_location_damped"] - df["user_avg_location"]).astype(np.float32)
    df["delta_sleep"]    = (df["hotel_avg_sleep_damped"]    - df["user_avg_sleep"]   ).astype(np.float32)

    print(f"{split_name:6s}: base features built — shape {df.shape}  in {time.time()-t0:.1f}s")
    return df


valid_df = build_base_features(valid_candidates_df, "valid")
test_df  = build_base_features(test_candidates_df,  "test")

del valid_candidates_df, test_candidates_df


valid : base features built — shape (50000000, 25)  in 5.2s
test  : base features built — shape (25000000, 25)  in 2.6s


## 7. Build sparse user-item matrix (support for co-visitation)


In [7]:
t0 = time.time()

# Node counts: max ids seen across train + eval candidates
all_users = pd.concat([train_df["user_idx"], valid_df["user_idx"], test_df["user_idx"]])
all_items = pd.concat([
    train_df["item_idx"],
    valid_df["candidate_item_idx"], test_df["candidate_item_idx"],
])
n_users = int(all_users.max()) + 1
n_items = int(all_items.max()) + 1

ones = np.ones(len(train_df), dtype=np.int8)
train_csr = sp.csr_matrix(
    (ones, (train_df["user_idx"].to_numpy(), train_df["item_idx"].to_numpy())),
    shape=(n_users, n_items),
    dtype=np.int8,
)
# Defensive: collapse any duplicate (user, item) entries back to binary
train_csr.sum_duplicates()
train_csr.data = np.ones_like(train_csr.data, dtype=np.int8)
train_csc = train_csr.tocsc()

# Per-item user counts — for cofreq normalization
item_user_counts = np.asarray(train_csr.sum(axis=0)).flatten().astype(np.int32)

# Per-user item lists
user_history = {}
for u, grp in train_df.groupby("user_idx"):
    user_history[int(u)] = grp["item_idx"].to_numpy(dtype=np.int64)

print(f"User-item CSR/CSC built: shape=({n_users:,}, {n_items:,})  nnz={train_csr.nnz:,}")
print(f"User history cached: {len(user_history):,} users")
print(f"Time: {time.time()-t0:.1f}s")


User-item CSR/CSC built: shape=(1,927,834, 309,904)  nnz=16,064,988
User history cached: 1,927,834 users
Time: 27.2s


## 8. Co-visitation features


In [8]:
def compute_cofreq_features(candidates_df, label):
    """Add candidate_cofreq_log and candidate_cofreq_normalized columns."""
    t0 = time.time()
    out_rows = len(candidates_df)
    cofreq_col      = np.zeros(out_rows, dtype=np.float32)
    cofreq_norm_col = np.zeros(out_rows, dtype=np.float32)

    df = candidates_df.reset_index(drop=True)
    grouped = df.groupby("user_idx", sort=False)
    n_groups = grouped.ngroups
    processed = 0

    for u, grp in grouped:
        u_int = int(u)
        cand_items = grp["candidate_item_idx"].to_numpy(dtype=np.int64)
        row_pos    = grp.index.to_numpy()

        hist = user_history.get(u_int, None)
        if hist is None or len(hist) == 0:
            processed += 1
            continue

        # Which users visited any of u's history items?
        hist_users_any = (np.asarray(train_csc[:, hist].sum(axis=1)).flatten() > 0).astype(np.int32)

        # Dot product against each candidate's user-column → per-candidate cofreq
        cand_cols = train_csc[:, cand_items]
        cofreq = np.asarray(hist_users_any @ cand_cols).flatten()

        cand_pop = item_user_counts[cand_items].astype(np.float32)
        cofreq_norm = cofreq / (cand_pop + COFREQ_NORM_EPS)

        cofreq_col[row_pos]      = np.log1p(cofreq).astype(np.float32)
        cofreq_norm_col[row_pos] = cofreq_norm.astype(np.float32)

        processed += 1
        if processed % 5000 == 0:
            print(f"  {label}: {processed:,}/{n_groups:,} users  elapsed={time.time()-t0:.0f}s")

    df["candidate_cofreq_log"]        = cofreq_col
    df["candidate_cofreq_normalized"] = cofreq_norm_col
    print(f"{label}: cofreq features done in {time.time()-t0:.1f}s")
    return df


valid_df = compute_cofreq_features(valid_df, "valid")
test_df  = compute_cofreq_features(test_df,  "test")

# Sanity check — positives should have higher cofreq than negatives
print("\nPositive vs negative cofreq (valid):")
print(valid_df.groupby("is_positive")[["candidate_cofreq_log", "candidate_cofreq_normalized"]].mean().round(4))


  valid: 5,000/50,000 users  elapsed=51s
  valid: 10,000/50,000 users  elapsed=102s
  valid: 15,000/50,000 users  elapsed=153s
  valid: 20,000/50,000 users  elapsed=205s
  valid: 25,000/50,000 users  elapsed=256s
  valid: 30,000/50,000 users  elapsed=308s
  valid: 35,000/50,000 users  elapsed=360s
  valid: 40,000/50,000 users  elapsed=412s
  valid: 45,000/50,000 users  elapsed=463s
  valid: 50,000/50,000 users  elapsed=514s
valid: cofreq features done in 514.2s
  test: 5,000/25,000 users  elapsed=51s
  test: 10,000/25,000 users  elapsed=102s
  test: 15,000/25,000 users  elapsed=154s
  test: 20,000/25,000 users  elapsed=206s
  test: 25,000/25,000 users  elapsed=257s
test: cofreq features done in 257.4s

Positive vs negative cofreq (valid):
             candidate_cofreq_log  candidate_cofreq_normalized
is_positive                                                   
0                          1.0610                       0.0133
1                          1.8324                       0.0323

## 9. Geographic features

Three features built here, all from `geo_id` and the user's training history.

- **`user_history_count_in_candidate_geo`** — number of times the user has reviewed a hotel in this candidate's geo (in train).
- **`candidate_geo_seen_by_user`** — binary version (1 if count > 0).



In [9]:
t0 = time.time()

# Per (user, geo) visit count from train
user_geo_counts = (
    train_df.merge(item_meta_raw[["item_idx", "geo_id"]], on="item_idx", how="left")
            .groupby(["user_idx", "geo_id"]).size()
            .reset_index(name="user_history_count_in_candidate_geo")
            .rename(columns={"geo_id": "candidate_geo_id"})
)
user_geo_counts["user_history_count_in_candidate_geo"] = (
    user_geo_counts["user_history_count_in_candidate_geo"].astype(np.int32)
)
print(f"Built user-geo count table: {len(user_geo_counts):,} rows  in {time.time()-t0:.1f}s")


def attach_geo_features(df, label):
    t0 = time.time()
    df = df.merge(
        user_geo_counts,
        on=["user_idx", "candidate_geo_id"], how="left",
    )
    df["user_history_count_in_candidate_geo"] = (
        df["user_history_count_in_candidate_geo"].fillna(0).astype(np.int32)
    )
    df["candidate_geo_seen_by_user"] = (
        (df["user_history_count_in_candidate_geo"] > 0).astype(np.int8)
    )
    print(f"{label}: geo features attached in {time.time()-t0:.1f}s")
    return df


valid_df = attach_geo_features(valid_df, "valid")
test_df  = attach_geo_features(test_df,  "test")

# candidate_geo_id was only needed for the geo merge
valid_df = valid_df.drop(columns=["candidate_geo_id"])
test_df  = test_df.drop(columns=["candidate_geo_id"])

# Sanity check
print("\nPositive vs negative geo features (valid):")
print(valid_df.groupby("is_positive")[
    ["candidate_geo_seen_by_user", "user_history_count_in_candidate_geo"]
].mean().round(4))


Built user-geo count table: 14,774,641 rows  in 4.4s
valid: geo features attached in 7.4s
test: geo features attached in 4.2s

Positive vs negative geo features (valid):
             candidate_geo_seen_by_user  user_history_count_in_candidate_geo
is_positive                                                                 
0                                0.0681                               0.0865
1                                0.1735                               0.2532


## 10. Final feature list and save

24 features ordered by section. NB3 (training) reads this column list and constructs ablation groups by section.


In [10]:
FEATURE_COLS = [
    # Section 1 — Retrieval signals
    "lightgcn_score",
    "lightgcn_rank",
    "lightgcn_score_zscore",
    # Section 2 — Hotel quality
    "hotel_review_count_log",
    "hotel_bayesian_avg",
    "hotel_avg_value_damped",
    "hotel_avg_location_damped",
    "hotel_avg_sleep_damped",
    # Section 3 — User profile
    "user_avg_rating",
    "user_review_count_log",
    "user_rating_std",
    # Section 4 — User aspect preferences (NEW in v3)
    "user_avg_value",
    "user_avg_location",
    "user_avg_sleep",
    # Section 5 — User × hotel interaction
    "candidate_rating_minus_user_avg",
    "candidate_pop_minus_user_avg_pop",
    "candidate_cofreq_log",
    "candidate_cofreq_normalized",
    # Section 6 — Aspect deltas (NEW in v3)
    "delta_value",
    "delta_location",
    "delta_sleep",
    # Section 7 — Geographic
    "candidate_geo_seen_by_user",
    "user_history_count_in_candidate_geo",
    "hotel_popularity_in_geo",
]
OUTPUT_COLS = BASE_COLS + FEATURE_COLS

# Verify everything is present before saving
missing_v = [c for c in OUTPUT_COLS if c not in valid_df.columns]
missing_t = [c for c in OUTPUT_COLS if c not in test_df.columns]
if missing_v or missing_t:
    raise RuntimeError(f"Missing columns — valid: {missing_v}  test: {missing_t}")

valid_df = valid_df[OUTPUT_COLS]
test_df  = test_df[OUTPUT_COLS]

print(f"Final feature count: {len(FEATURE_COLS)}")
for i, f in enumerate(FEATURE_COLS, 1):
    print(f"  {i:2d}. {f}")

valid_df.to_parquet(VALID_FEATURES_FILE, index=False)
test_df.to_parquet(TEST_FEATURES_FILE,  index=False)

for path in [VALID_FEATURES_FILE, TEST_FEATURES_FILE]:
    size_mb = os.path.getsize(path) / 1024**2
    print(f"Saved {path}: {size_mb:.1f} MB")


Final feature count: 24
   1. lightgcn_score
   2. lightgcn_rank
   3. lightgcn_score_zscore
   4. hotel_review_count_log
   5. hotel_bayesian_avg
   6. hotel_avg_value_damped
   7. hotel_avg_location_damped
   8. hotel_avg_sleep_damped
   9. user_avg_rating
  10. user_review_count_log
  11. user_rating_std
  12. user_avg_value
  13. user_avg_location
  14. user_avg_sleep
  15. candidate_rating_minus_user_avg
  16. candidate_pop_minus_user_avg_pop
  17. candidate_cofreq_log
  18. candidate_cofreq_normalized
  19. delta_value
  20. delta_location
  21. delta_sleep
  22. candidate_geo_seen_by_user
  23. user_history_count_in_candidate_geo
  24. hotel_popularity_in_geo
Saved phase3_outputs\valid_lambdamart_features.parquet: 2378.0 MB
Saved phase3_outputs\test_lambdamart_features.parquet: 1189.3 MB


---

# Diagnostics

## Univariate NDCG@K

In [11]:
def univariate_ndcg(df, feature_cols, k):
    """Sort each user's candidates by the feature alone and compute NDCG@k."""
    results = []
    n_users = df["user_idx"].nunique()
    for f in feature_cols:
        ascending = (f == "lightgcn_rank")
        sorted_df = df.sort_values(
            ["user_idx", f],
            ascending=[True, ascending],
            kind="stable",
        )
        sorted_df["_pos"] = sorted_df.groupby("user_idx").cumcount()
        hits = sorted_df[
            (sorted_df["is_positive"] == 1) & (sorted_df["_pos"] < k)
        ]
        ndcg_contrib = 1.0 / np.log2(hits["_pos"].to_numpy() + 2)
        results.append({
            "feature": f,
            "direction": "asc" if ascending else "desc",
            f"HR@{k}": len(hits) / n_users,
            f"NDCG@{k}": float(ndcg_contrib.sum() / n_users),
        })
    return pd.DataFrame(results).sort_values(f"NDCG@{k}", ascending=False)


print(f"Univariate NDCG@{UNIVARIATE_NDCG_K} on valid (sorted best to worst)")
univariate_valid = univariate_ndcg(valid_df, FEATURE_COLS, UNIVARIATE_NDCG_K)
print(univariate_valid.round(4).to_string(index=False))


Univariate NDCG@10 on valid (sorted best to worst)
                            feature direction  HR@10  NDCG@10
               candidate_cofreq_log      desc 0.0279   0.0149
        candidate_cofreq_normalized      desc 0.0185   0.0098
                     lightgcn_score      desc 0.0192   0.0097
                      lightgcn_rank       asc 0.0192   0.0097
              user_review_count_log      desc 0.0192   0.0097
                    user_avg_rating      desc 0.0192   0.0097
              lightgcn_score_zscore      desc 0.0192   0.0097
                  user_avg_location      desc 0.0192   0.0097
                    user_rating_std      desc 0.0192   0.0097
                     user_avg_sleep      desc 0.0192   0.0097
                     user_avg_value      desc 0.0192   0.0097
user_history_count_in_candidate_geo      desc 0.0183   0.0096
         candidate_geo_seen_by_user      desc 0.0182   0.0094
             hotel_review_count_log      desc 0.0061   0.0028
   candidate_pop_mi

## Feature correlation


In [12]:
corr = valid_df[FEATURE_COLS].corr()

pairs = []
for i in range(len(FEATURE_COLS)):
    for j in range(i + 1, len(FEATURE_COLS)):
        c = corr.iloc[i, j]
        pairs.append({
            "f1": FEATURE_COLS[i],
            "f2": FEATURE_COLS[j],
            "corr": c,
            "abs_corr": abs(c),
        })
pairs_df = pd.DataFrame(pairs).sort_values("abs_corr", ascending=False)
print(f"Top {N_CORR_TOP_PAIRS} correlated feature pairs (valid)")
print(pairs_df.head(N_CORR_TOP_PAIRS).round(3).to_string(index=False))


Top 15 correlated feature pairs (valid)
                        f1                                  f2    corr  abs_corr
             lightgcn_rank               lightgcn_score_zscore -0.8980    0.8980
        hotel_bayesian_avg              hotel_avg_sleep_damped  0.8860    0.8860
    hotel_avg_value_damped                         delta_value  0.8520    0.8520
 hotel_avg_location_damped                      delta_location  0.8410    0.8410
candidate_geo_seen_by_user user_history_count_in_candidate_geo  0.8390    0.8390
    hotel_avg_sleep_damped                         delta_sleep  0.8150    0.8150
        hotel_bayesian_avg              hotel_avg_value_damped  0.8100    0.8100
           user_avg_rating     candidate_rating_minus_user_avg -0.8060    0.8060
               delta_value                         delta_sleep  0.7420    0.7420
            user_avg_value                      user_avg_sleep  0.7190    0.7190
    hotel_avg_value_damped              hotel_avg_sleep_damped  0.718

## Positive vs negative — Cohen's d


In [13]:
def pos_neg_stats(df, feature_cols, split_name):
    pos = df[df["is_positive"] == 1]
    neg = df[df["is_positive"] == 0]
    rows = []
    for f in feature_cols:
        pm, nm = float(pos[f].mean()), float(neg[f].mean())
        ps, ns = float(pos[f].std()),  float(neg[f].std())
        pooled = np.sqrt((ps**2 + ns**2) / 2) if (ps > 0 or ns > 0) else 1.0
        d = (pm - nm) / pooled if pooled > 0 else 0.0
        rows.append({
            "split": split_name, "feature": f,
            "neg_mean": nm, "pos_mean": pm,
            "diff": pm - nm, "cohens_d": d,
        })
    return pd.DataFrame(rows)


pos_neg_df = pos_neg_stats(valid_df, FEATURE_COLS, "valid")
print("Positive vs negative (valid)")
print("Cohen's d rough guide: |d|>0.2 small, >0.5 medium, >0.8 large")
print(pos_neg_df.sort_values("cohens_d", key=lambda s: s.abs(), ascending=False).round(4).to_string(index=False))


Positive vs negative (valid)
Cohen's d rough guide: |d|>0.2 small, >0.5 medium, >0.8 large
split                             feature  neg_mean  pos_mean      diff  cohens_d
valid                candidate_cofreq_log    1.0610    1.8324    0.7714    0.6842
valid               lightgcn_score_zscore   -0.0002    0.8039    0.8041    0.6357
valid                       lightgcn_rank  499.5503  329.1377 -170.4126   -0.5889
valid         candidate_cofreq_normalized    0.0133    0.0323    0.0190    0.4399
valid                      lightgcn_score    7.7037    8.5334    0.8297    0.4343
valid          candidate_geo_seen_by_user    0.0681    0.1735    0.1054    0.3278
valid user_history_count_in_candidate_geo    0.0865    0.2532    0.1667    0.3024
valid              hotel_review_count_log    5.3780    5.6741    0.2961    0.2652
valid    candidate_pop_minus_user_avg_pop    0.4362    0.6169    0.1807    0.1776
valid              hotel_avg_sleep_damped    4.1727    4.2096    0.0369    0.0989
valid  